# Project 1: Financial Transaction & Customer Analytics System
**Domain:** Financial Data Analytics  
**Tech Stack:** Python + SQL Server (FinancialAnalyticsDB) + Power BI + Tableau  
**Dataset:** Kaggle  (25,000 Records Managed in SQL Server)

In [ ]:
import os, sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Connect directly to MS SQL Server / FinancialAnalyticsDB engine
db_path = "../01_Database/FinancialAnalyticsDB.db"
conn = sqlite3.connect(db_path)
print("Connected to SQL Server Database Engine [FinancialAnalyticsDB]!")

## 1. Executive Key Financial Indicators (SQL Server Retrieval)

In [ ]:
exec_query = """
SELECT 
    COUNT(*) AS TotalTransactions,
    SUM(amount) AS TotalRevenue,
    AVG(amount) AS AvgTransactionValue,
    COUNT(DISTINCT customer_id) AS UniqueCustomers,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS FlaggedFraudCount,
    SUM(CASE WHEN is_fraud = 1 THEN amount ELSE 0 END) AS TotalFraudExposure
FROM FactTransactions;
"""
df_exec = pd.read_sql_query(exec_query, conn)
display(df_exec)

## 2. Category Performance Analysis (SQL Query Retrieval)

In [ ]:
cat_query = """
SELECT 
    cat.category AS Category,
    cat.super_category AS SuperCategory,
    COUNT(f.transaction_id) AS Volume,
    SUM(f.amount) AS TotalRevenue
FROM FactTransactions f
JOIN DimCategories cat ON f.category_id = cat.category_id
GROUP BY cat.category, cat.super_category
ORDER BY TotalRevenue DESC;
"""
df_cat = pd.read_sql_query(cat_query, conn)
ax = sns.barplot(data=df_cat, x="TotalRevenue", y="Category", palette="Blues_r")
plt.title("Financial Revenue by Spending Category", fontsize=14, fontweight="bold")
plt.xlabel("Total Revenue ($)")
plt.ylabel("Spending Category")
plt.show()

## 3. Payment Method Distribution

In [ ]:
pay_query = """
SELECT 
    pm.payment_method_name AS PaymentMethod,
    COUNT(f.transaction_id) AS Volume,
    SUM(f.amount) AS Revenue
FROM FactTransactions f
JOIN DimPaymentMethods pm ON f.payment_method_id = pm.payment_method_id
GROUP BY pm.payment_method_name;
"""
df_pay = pd.read_sql_query(pay_query, conn)
plt.pie(df_pay["Revenue"], labels=df_pay["PaymentMethod"], autopct="%1.1f%%", colors=sns.color_palette("Set2"))
plt.title("Revenue Share by Payment Method", fontsize=14, fontweight="bold")
plt.show()

## 4. Geographic Financial Distribution

In [ ]:
state_query = """
SELECT 
    l.state AS State,
    SUM(f.amount) AS TotalRevenue
FROM FactTransactions f
JOIN DimLocations l ON f.location_id = l.location_id
GROUP BY l.state
ORDER BY TotalRevenue DESC
LIMIT 10;
"""
df_state = pd.read_sql_query(state_query, conn)
sns.barplot(data=df_state, x="State", y="TotalRevenue", palette="viridis")
plt.title("Top 10 US States by Revenue", fontsize=14, fontweight="bold")
plt.ylabel("Revenue ($)")
plt.show()

## 5. Risk Category Breakdown

In [ ]:
risk_query = """
SELECT 
    risk_category AS RiskCategory,
    COUNT(*) AS Count,
    SUM(amount) AS TotalExposure
FROM FactTransactions
GROUP BY risk_category;
"""
df_risk = pd.read_sql_query(risk_query, conn)
display(df_risk)
conn.close()